In [1]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

'/opt/spark'

In [2]:
from datetime import datetime, timedelta
import sys

from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
                    .master("yarn") \
                    .appName(f"EventsPartitioningJob") \
                    .getOrCreate()



SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]


26/09/08 17:20:12 WARN Utils: Your hostname, fv4t6etie0mtambli5jk resolves to a loopback address: 127.0.1.1; using 10.130.0.42 instead (on interface eth0)
26/09/08 17:20:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/08 17:20:13 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/09/08 17:20:18 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/09/08 17:20:18 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [4]:
events_dir = "/user/master/data/geo/events"
geo_path = "/user/s18314377/data/geo/geo.csv"

In [5]:
## Шаг 2

def clean_coord(col_name):
    return F.regexp_replace(F.col(col_name), ",", ".").cast(DoubleType())

def get_messages_closest_cities(df_messages, df_cities):
    df_cities_prepared = df_cities.select(
        F.col("city"),
        F.col("lat").alias("city_lat"),
        F.col("lng").alias("city_lon")
    )

    df_messages_prepared = df_messages.select(
        "event",
        "event_type",
        "date",
        F.col("lat").alias("event_lat"),
        F.col("lon").alias("event_lon")
    )

    df_joined = df_messages_prepared.crossJoin(F.broadcast(df_cities_prepared))

    lat1 = F.radians(F.col("event_lat"))
    lon1 = F.radians(F.col("event_lon"))
    lat2 = F.radians(clean_coord("city_lat"))
    lon2 = F.radians(clean_coord("city_lon"))
    
    r = 6371

    distance_formula = (
        F.lit(2 * r) * F.asin(
            F.sqrt(    
                F.pow(F.sin((lat2 - lat1) / 2), 2) + 
                F.cos(lat1) * F.cos(lat2) * F.pow(F.sin((lon2 - lon1) / 2), 2)
            )
        )
    )

    df_joined = df_joined.withColumn("distance", distance_formula)
    
    window = Window().partitionBy('event.message_id').orderBy("distance")

    df_messages_closest_cities = df_joined \
                .withColumn("row_number", F.row_number().over(window)) \
                .filter(F.col("row_number") == 1) \
                .drop("row_number", "city_lat", "city_lon")
    return df_messages_closest_cities
    
    
def get_users_act_cities(df_messages_closest_cities):
    df_act_city = df_messages_closest_cities \
        .withColumn(
            "rank_act_city", 
            F.row_number().over(
                Window().partitionBy("event.message_from").orderBy(F.col("event.message_ts").desc())
            )
        ) \
        .filter(F.col("rank_act_city") == 1) \
        
    return df_act_city.select(F.col("event.message_from").alias("user_id"), F.col("city").alias("act_city"))


def get_users_home_cities(df_messages_closest_cities, days_count=27):
    df_unique_days = df_messages_closest_cities.select(
        F.col("event.message_from").alias("user_id"),
        "city",
        "date"
    ).distinct()
    
    window_user_city = Window.partitionBy("user_id", "city").orderBy("date")
    
    # Сделали нумерацию в группах по юзерам-пользователям для каждого уникального дня
    df_indexed_days = df_unique_days.withColumn("day_index", F.row_number().over(window_user_city))
    
    # Вычитаем из дат дни, равные индексу 'day_seq'. При пропуске даты значение изменится - новая непрерывная группа
    df_continuous_sequences = df_indexed_days.withColumn("seq_start_date", F.date_sub(F.col("date"), F.col("day_index")))
    
    # Группируем по условным "датам начала последовательностей"
    df_continuous_groups = (
        df_continuous_sequences
        .groupBy("user_id", "city", "seq_start_date")
        .agg(F.count("date").alias("continuous_days"))
    )

    # Фильтруем по нужному days_count 
    df_filtered = df_continuous_groups.filter(F.col("continuous_days") >= days_count)
    
    window_latest = Window.partitionBy("user_id").orderBy(F.col("seq_start_date").desc())

    df_result = (
        df_filtered
        .withColumn("island_rank", F.row_number().over(window_latest))
        .filter(F.col("island_rank") == 1)
    )
    
    return df_result.select("user_id", F.col("city").alias("home_city"))



def get_users_travel_info(df_messages_closest_cities):
    # Шаг 1. Оставляем только уникальные дни (user + date), чтобы избавиться от дублей сообщений в один день
    df_user_days = df_messages_closest_cities.select(
        F.col("event.message_from").alias("user_id"), 
        "date", 
        "city"
    ).distinct()

    # Шаг 2. Окно для отслеживания хронологии пользователя
    window_time = Window.partitionBy("user_id").orderBy("date")

    # Шаг 3. Находим город из предыдущего дня
    df_with_prev = df_user_days.withColumn("prev_city", F.lag("city", 1).over(window_time))

    # Шаг 4. Ставим 1, если пользователь переехал в другой город (или это его первая запись)
    df_with_marker = df_with_prev.withColumn(
        "is_new_travel",
        F.when(F.col("prev_city").isNull() | (F.col("city") != F.col("prev_city")), 1).otherwise(0)
    )

    # Шаг 5. Создаем уникальный ID путешествия с помощью кумулятивной суммы
    # (Для каждого нового визита это число будет увеличиваться: 1, 2, 3...)
    df_with_travel_id = df_with_marker.withColumn(
        "travel_id", 
        F.sum("is_new_travel").over(window_time)
    )

    # Шаг 6. Схлопываем дни внутри одного путешествия. Нам нужна дата начала, чтобы сохранить хронологию визитов
    df_visits = (
        df_with_travel_id
        .groupBy("user_id", "travel_id", "city")
        .agg(F.min("date").alias("travel_start_date"))
    )

    # Шаг 7. Финальное окно, чтобы собрать города по порядку начала путешествий
    window_collect = Window.partitionBy("user_id").orderBy("travel_start_date")

    df_final_metrics = (
        df_visits
        # Собираем массив городов в правильном порядке
        .withColumn("travel_array", F.collect_list("city").over(window_collect))
        # Считаем общее количество посещений (размер массива)
        .withColumn("travel_count", F.size("travel_array"))
        # Берем последнюю строку для каждого юзера, где массивы собраны полностью
        .withColumn("row_num", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("travel_start_date"))))
        .filter(F.col("row_num") == 1)
        .select("user_id", "travel_count", "travel_array")
    )

    # Смотрим результат
    return df_final_metrics



def get_user_local_time(df_messages_closest_cities):

    city_to_tz_map = F.create_map([
        # Сидней
        F.lit("Sydney"), F.lit("Australia/Sydney"),
        F.lit("Canberra"), F.lit("Australia/Sydney"),
        F.lit("Newcastle"), F.lit("Australia/Sydney"),
        F.lit("Wollongong"), F.lit("Australia/Sydney"),
        F.lit("Maitland"), F.lit("Australia/Sydney"),
    
        # Мельбурн
        F.lit("Melbourne"), F.lit("Australia/Melbourne"),
        F.lit("Cranbourne"), F.lit("Australia/Melbourne"),
        F.lit("Geelong"), F.lit("Australia/Melbourne"),
        F.lit("Ballarat"), F.lit("Australia/Melbourne"),
        F.lit("Bendigo"), F.lit("Australia/Melbourne"),
    
        # Брисбен
        F.lit("Brisbane"), F.lit("Australia/Brisbane"),
        F.lit("Gold Coast"), F.lit("Australia/Brisbane"),
        F.lit("Townsville"), F.lit("Australia/Brisbane"),
        F.lit("Ipswich"), F.lit("Australia/Brisbane"),
        F.lit("Cairns"), F.lit("Australia/Brisbane"),
        F.lit("Toowoomba"), F.lit("Australia/Brisbane"),
        F.lit("Mackay"), F.lit("Australia/Brisbane"),
        F.lit("Rockhampton"), F.lit("Australia/Brisbane"),
    
        # Перт
        F.lit("Perth"), F.lit("Australia/Perth"),
        F.lit("Bunbury"), F.lit("Australia/Perth"),
    
        # Аделаида
        F.lit("Adelaide"), F.lit("Australia/Adelaide"),
    
        # Хобарт
        F.lit("Hobart"), F.lit("Australia/Hobart"),
        F.lit("Launceston"), F.lit("Australia/Hobart"),
    
        # Дарвин
        F.lit("Darwin"), F.lit("Australia/Darwin")
    ])



    df_local_time = (
        df_messages_closest_cities
        .withColumn(
            "TIME_UTC", 
            F.date_format(
                F.coalesce(
                    F.col("event.message_ts"), 
                    F.col("event.datetime")
                ).cast("timestamp"),
                "yyyy-MM-dd HH:mm:ss"
            ).cast("timestamp")
        )
        .withColumn(
            "timezone",  
            F.coalesce(
                city_to_tz_map[F.initcap(F.trim(F.col("city")))], 
                F.lit("Australia/Sydney") 
            )
        )
    )

    df_local_time_latest = (
        df_local_time
        .select(F.col("event.message_from").alias("user_id"), "TIME_UTC", "timezone")
        .withColumn("local_time", F.from_utc_timestamp(F.col("TIME_UTC"),F.col('timezone')))
        .withColumn(
            "rank", 
            F.row_number().over(
                Window().partitionBy("user_id").orderBy(F.col("TIME_UTC").desc()))
        )
        .filter(F.col("rank") == 1)
        .drop("rank", "TIME_UTC", "timezone")
    )
    
    return df_local_time_latest


In [6]:
def main(events_dir, geo_path, home_days_count=27):
    df_events = (
        spark.read
        .option("pathGlobFilter", "*part-000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )
    df_messages = df_events.filter(F.col("event_type") == "message")
    df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)
    
    ### Этап 1. Ищем ближайшие города к каждому сообщению
    df_messages_closest_cities = get_messages_closest_cities(df_messages, df_cities)
    
    ### Этап 2. Ищем последний город пользователя и последний домашний город пользователя
    df_act_cities = get_users_act_cities(df_messages_closest_cities)
    df_home_cities = get_users_home_cities(df_messages_closest_cities, days_count=home_days_count)
    
    ### Этап 3. Ищем статистику путешествий
    df_travel_info = get_users_travel_info(df_messages_closest_cities)
    
    
    ### Этап 4. Ищем местное время
    df_local_time = get_user_local_time(df_messages_closest_cities)
    
    
    df_result = (
        df_act_cities
        .join(df_home_cities, on="user_id", how="outer")
        .join(df_travel_info, on="user_id", how="outer")
        .join(df_local_time, on="user_id", how="outer")
    )
    return df_result
    

In [7]:
# df_res = main(events_dir, geo_path, home_days_count=3)

In [8]:
# df_res.cache()

In [9]:
# df_res.orderBy("user_id").show(1000)

In [29]:
df_events = (
        spark.read
        .option("pathGlobFilter", "*part-0000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )

df_events = df_events.sample(withReplacement=False, fraction=0.1, seed=42)
# df_events = df_events.filter(F.col("date") == "2022-01-01")


df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)

df_events_closest_cities = get_messages_closest_cities(df_events, df_cities)




In [30]:
df_zone_stats = (
    df_events_closest_cities
    .withColumn("month", F.month(F.col("date")))
    .withColumn("week", F.weekofyear(F.col("date")))
    .withColumn(
        "first_message_ts", 
        F.min(F.when(F.col("event_type") == "message", F.col("event.message_ts"))).over(Window.partitionBy("event.message_from"))
    )

)


In [31]:
df_zone_stats.cache()

DataFrame[event: struct<admins:array<bigint>,channel_id:bigint,datetime:string,media:struct<media_type:string,src:string>,message:string,message_channel_to:bigint,message_from:bigint,message_group:bigint,message_id:bigint,message_to:bigint,message_ts:string,reaction_from:string,reaction_type:string,subscription_channel:bigint,subscription_user:string,tags:array<string>,user:string>, event_type: string, date: date, event_lat: double, event_lon: double, city: string, distance: double, month: int, week: int, first_message_ts: string]

In [32]:
df_zone_stats.printSchema()

root
 |-- event: struct (nullable = true)
 |    |-- admins: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- channel_id: long (nullable = true)
 |    |-- datetime: string (nullable = true)
 |    |-- media: struct (nullable = true)
 |    |    |-- media_type: string (nullable = true)
 |    |    |-- src: string (nullable = true)
 |    |-- message: string (nullable = true)
 |    |-- message_channel_to: long (nullable = true)
 |    |-- message_from: long (nullable = true)
 |    |-- message_group: long (nullable = true)
 |    |-- message_id: long (nullable = true)
 |    |-- message_to: long (nullable = true)
 |    |-- message_ts: string (nullable = true)
 |    |-- reaction_from: string (nullable = true)
 |    |-- reaction_type: string (nullable = true)
 |    |-- subscription_channel: long (nullable = true)
 |    |-- subscription_user: string (nullable = true)
 |    |-- tags: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |   

In [34]:
def get_grouped_stats(df_zone_stats):

    df_stats_month = (
        df_zone_stats
        .groupBy("month", F.col("city").alias("zone_id"))
        .agg(
            F.sum(F.when(F.col("event_type") == "message", 1).otherwise(0)).alias("month_message"),
            F.sum(F.when(F.col("event_type") == "reaction", 1).otherwise(0)).alias("month_reaction"),        
            F.sum(F.when(F.col("event_type") == "subscription", 1).otherwise(0)).alias("month_subscription"),
            F.sum(
                F.when(
                    (F.col("event.message_ts") == F.col("first_message_ts")), 
                    1
                ).otherwise(0)
            ).alias("month_user")
        )
    )


    df_stats_week = (
    df_zone_stats
    .groupBy("month", "week", F.col("city").alias("zone_id"))  # Месяц здесь нужен как мостик для джойна
    .agg(
        F.sum(F.when(F.col("event_type") == "message", 1).otherwise(0)).alias("week_message"),
        F.sum(F.when(F.col("event_type") == "reaction", 1).otherwise(0)).alias("week_reaction"),
        F.sum(F.when(F.col("event_type") == "subscription", 1).otherwise(0)).alias("week_subscription"),
        F.sum(
                F.when(
                    (F.col("event.message_ts") == F.col("first_message_ts")), 
                    1
                ).otherwise(0)
            ).alias("month_user")
        
        )
    )
    final_df = df_stats_week.join(
        df_stats_month, 
        on=["month", "zone_id"], 
        how="left"
    )
    return final_df

In [35]:
df_stats = get_grouped_stats(df_zone_stats)

In [36]:
df_stats.show(100)

+-----+-----------+----+------------+-------------+-----------------+----------+-------------+--------------+------------------+----------+
|month|    zone_id|week|week_message|week_reaction|week_subscription|month_user|month_message|month_reaction|month_subscription|month_user|
+-----+-----------+----+------------+-------------+-----------------+----------+-------------+--------------+------------------+----------+
|    3|     Darwin|  11|           0|           22|                0|         0|            1|            86|                 0|         1|
|    2|     Mackay|   7|           0|           13|                0|         0|            0|            37|                 0|         0|
|    2|     Mackay|   6|           0|           11|                0|         0|            0|            37|                 0|         0|
|    2|      Perth|   5|           0|           10|                0|         0|            0|            41|                 0|         0|
|    4|    Ipswich| 

In [37]:
def main_2():
    df_events = (
        spark.read
        .option("pathGlobFilter", "*part-0000*.parquet") # Читаем только самый первый под-файл в каждой папке
        .parquet(events_dir)     # Заходим во все партиции
    )

    df_events = df_events.sample(withReplacement=False, fraction=0.1, seed=42)
    # df_events = df_events.filter(F.col("date") == "2022-01-01")


    df_cities = spark.read.csv(geo_path, sep=";", header=True, inferSchema=True)

    df_events_closest_cities = get_messages_closest_cities(df_events, df_cities)
    
    df_stats = get_grouped_stats(df_zone_stats)
    
    return df_stats
    

In [38]:
df_stats = main_2()

In [39]:
df_stats.cache()

DataFrame[month: int, zone_id: string, week: int, week_message: bigint, week_reaction: bigint, week_subscription: bigint, month_user: bigint, month_message: bigint, month_reaction: bigint, month_subscription: bigint, month_user: bigint]

In [43]:
df_stats.orderBy("month", "week", "zone_id").show(100)

+-----+-----------+----+------------+-------------+-----------------+----------+-------------+--------------+------------------+----------+
|month|    zone_id|week|week_message|week_reaction|week_subscription|month_user|month_message|month_reaction|month_subscription|month_user|
+-----+-----------+----+------------+-------------+-----------------+----------+-------------+--------------+------------------+----------+
|    1|   Adelaide|   1|           2|            0|                0|         2|            2|             7|                 0|         2|
|    1| Launceston|   1|           1|            0|                0|         1|            1|             4|                 0|         1|
|    1|  Newcastle|   1|           1|            0|                0|         1|            1|            10|                 0|         1|
|    1|   Brisbane|   2|           1|            0|                0|         0|            1|            15|                 0|         0|
|    1|   Adelaide| 

In [ ]:
df_zone_stats.select(
    ,
    "week",
    "month",
    "event_type",
    "event.message_id",
    "event.subscription_channel",
    "event.subscription_user",
    "event.reaction_from",
    "event.reaction_type"
).show(100)